# 03 Detect Symbols: Stage 9

Finds signal symbols on a SIP via two detectors:

- **Template matching** (OpenCV `matchTemplate`, multi-scale) for the simple distinctive shapes: KM marker, S/B station-building box, BSLB, GL gate-lodge box.
- **DINOv2 nearest-neighbor** (`facebook/dinov2-base` + cosine similarity over exemplar embeddings) for the chained-ellipse signal posts: Distant, Home, Starter, Shunt, Calling-on.

Requires the symbol library to be built first (`notebooks/02_build_library.ipynb` -> `library/index.json`).

This notebook re-runs Stages 1-7 (preprocessing + OCR + classify) on the SIP for self-containment, then runs Stage 9. On Colab T4 the full notebook takes roughly: preprocess 30s, OCR ~2 min, symbols ~30-60s. CPU is much slower for OCR and DINOv2.

## Setup

Detect Colab vs local. On Colab, mount Drive and point model caches at `.model_cache/` so PaddleOCR / DINOv2 weights persist across sessions. Locally, `PROJECT_ROOT` is the repo root.

In [ ]:
import os
import sys
from pathlib import Path

# Disable paddle's oneDNN backend and PIR executor before any paddle import.
# Same workaround as 01_preprocess.ipynb.
os.environ.setdefault("FLAGS_use_mkldnn", "0")
os.environ.setdefault("FLAGS_enable_pir_in_executor", "0")

IS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/sip-extractor")
    cache = PROJECT_ROOT / ".model_cache"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache / "huggingface")
    os.environ["TORCH_HOME"] = str(cache / "torch")
    os.environ["PADDLE_HOME"] = str(cache / "paddle")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working dir: {PROJECT_ROOT}")
print(f"Colab: {IS_COLAB}")

In [ ]:
# Colab-only: pull the latest repo state from GitHub.
if IS_COLAB:
    print("Pulling latest from GitHub...")
    !git -C {PROJECT_ROOT} pull --ff-only origin main 2>&1 | tail -5
    print()
    print("If the pull added or modified notebook cells, close and reopen this")
    print("notebook tab to see them. Existing .py module changes load on next import.")

In [ ]:
# Colab-only: install runtime deps for Stages 1-7 + Stage 9.
#
# Picks paddle build by runtime. Same logic as 01_preprocess.ipynb.
# Plus reinstalls torch from PyPI (01_preprocess uninstalls it on Colab GPU
# to dodge a NCCL ABI mismatch; we put a fresh wheel back here so DINOv2
# can run). Latest stable torch from PyPI ships with a bundled libnccl
# matching its own ABI, sidestepping the Colab-image issue.
import importlib
import importlib.util
import shutil
import subprocess
import sys


def _has_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True, timeout=5)
        return True
    except Exception:
        return False


def _on_disk(mod: str) -> bool:
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False


def _module_present(mod: str) -> bool:
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


HAS_GPU = _has_gpu()
PADDLE_PKG = "paddlepaddle-gpu==3.0.0" if HAS_GPU else "paddlepaddle==3.0.0"
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/" if HAS_GPU else None

REQUIRED = {
    "fitz": "pymupdf",
    "cv2": "opencv-python",
    "skimage": "scikit-image",
    "paddleocr": "paddleocr==3.5.0",
    "paddle": PADDLE_PKG,
    "langchain_text_splitters": "langchain-text-splitters",
    "transformers": "transformers",
}

# Install torch on Colab. 01_preprocess.ipynb uninstalls torch on Colab GPU
# (Colab's bundled torch had a broken libnccl ABI). Here we ensure a fresh
# PyPI torch is installed so DINOv2 can run. -U upgrades if older, no-ops
# if already current; fast path when torch is already healthy.
if IS_COLAB:
    if not _on_disk("torch"):
        print("Installing torch from PyPI (~800MB, ~1-2 minutes on first run per VM)...")
        %pip install -q torch
    else:
        # torch already on disk: confirm it imports cleanly. If not, force-reinstall.
        if not _module_present("torch"):
            print("torch is on disk but failed to import; force-reinstalling from PyPI...")
            !pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3
            %pip install -q torch

# Same paddle build/runtime mismatch handling as 01_preprocess.
force_paddle_install = False
if _module_present("paddle"):
    import paddle as _p
    _has_cuda = _p.is_compiled_with_cuda()
    if HAS_GPU and not _has_cuda:
        print("Installed paddle is CPU build but runtime has GPU. Swapping...")
        !pip uninstall -y paddlepaddle paddlepaddle-gpu 2>&1 | tail -3
        force_paddle_install = True
    elif (not HAS_GPU) and _has_cuda:
        print("Installed paddle is GPU build but runtime is CPU. Swapping...")
        !pip uninstall -y paddlepaddle paddlepaddle-gpu 2>&1 | tail -3
        force_paddle_install = True
    else:
        REQUIRED.pop("paddle", None)

missing = [pkg for mod, pkg in REQUIRED.items() if not _module_present(mod)]
if force_paddle_install and PADDLE_PKG not in missing:
    missing.append(PADDLE_PKG)

if IS_COLAB and missing:
    print(f"Runtime: {'GPU' if HAS_GPU else 'CPU'}  Installing: {missing}")
    if HAS_GPU and any("paddlepaddle-gpu" in m for m in missing):
        gpu_pkgs = [m for m in missing if "paddlepaddle-gpu" in m]
        other_pkgs = [m for m in missing if "paddlepaddle-gpu" not in m]
        %pip install -q --index-url {PADDLE_INDEX} --extra-index-url https://pypi.org/simple {" ".join(gpu_pkgs)}
        if other_pkgs:
            %pip install -q {" ".join(other_pkgs)}
    else:
        %pip install -q {" ".join(missing)}
else:
    print(f"All deps present (IS_COLAB={IS_COLAB}, GPU={HAS_GPU})")

try:
    import paddle
    print(f"paddle {paddle.__version__}  CUDA build: {paddle.is_compiled_with_cuda()}  GPUs: {paddle.device.cuda.device_count() if paddle.is_compiled_with_cuda() else 0}")
except Exception as e:
    print(f"paddle import failed: {e}")

try:
    import torch
    print(f"torch {torch.__version__}  CUDA available: {torch.cuda.is_available()}  device count: {torch.cuda.device_count()}")
except Exception as e:
    print(f"torch import failed: {e}")


## Configure inputs and outputs

Point `PDF_PATH` at the SIP. Outputs go to `outputs/<sip_stem>/`. The symbol library is read from `library/` at the project root.

In [ ]:
TARGET_DPI = 300

PDF_PATH = None  # set this by hand to force a specific file

if PDF_PATH is None:
    candidates = sorted((PROJECT_ROOT / "data" / "sips").glob("*.pdf"))
    if not candidates:
        candidates = sorted(PROJECT_ROOT.glob("*.pdf"))
    if candidates:
        PDF_PATH = candidates[0]
        if len(candidates) > 1:
            print(f"Multiple PDFs found, using first. All: {[p.name for p in candidates]}")

assert PDF_PATH and PDF_PATH.exists(), (
    f"No SIP PDF found. Drop one in {PROJECT_ROOT/'data'/'sips'}/ "
    f"or set PDF_PATH explicitly."
)
OUT_DIR = PROJECT_ROOT / "outputs" / PDF_PATH.stem
LIB_DIR = PROJECT_ROOT / "library"

assert (LIB_DIR / "index.json").exists(), (
    f"library/index.json not found at {LIB_DIR}. "
    f"Run notebooks/02_build_library.ipynb first."
)

print(f"Input  : {PDF_PATH}")
print(f"Output : {OUT_DIR}")
print(f"Library: {LIB_DIR}")

## Stages 1-7: re-run preprocessing, OCR, classify

Self-contained: re-runs the prior pipeline so this notebook works without depending on `01_preprocess` artifacts. Cheap when artifacts already exist (PaddleOCR's first init still costs ~30s for the model load even with weights cached).

In [ ]:
from IPython.display import Image as IPyImage
from sip_extractor import preprocessing, ocr, classify

prep = preprocessing.run(PDF_PATH, OUT_DIR, target_dpi=TARGET_DPI)
print(f"Crop x=[{prep.crop_x_min}, {prep.crop_x_max}]")

text_entities = ocr.run(prep.gray_cropped, OUT_DIR)
print(f"OCR detections: {len(text_entities)}")

classify.run(text_entities, OUT_DIR, gray_cropped=prep.gray_cropped)

## Stage 9: detect symbols

Runs both template matching and DINOv2 NN. First run downloads ~340 MB of DINOv2 weights to `HF_HOME`.

In [ ]:
from sip_extractor import symbols

library = symbols.Library.from_path(LIB_DIR)
print(f"Library: {len(library.classes)} classes")

detections = symbols.run(
    prep.binary_cropped,
    text_entities,
    library,
    OUT_DIR,
    methods=("template_match", "dinov2_nn"),
)

print(f"\nDetected {len(detections)} symbols")
from collections import Counter
breakdown = Counter((s["class_name"], s["method"]) for s in detections)
for (cls, method), n in breakdown.most_common():
    print(f"  {method:14s} {cls:30s}: {n}")

IPyImage(str(OUT_DIR / "symbols_overlay_preview.png"))

## Outputs

After this notebook runs, `outputs/<sip_stem>/` contains the Stage 6/7 outputs from `01_preprocess` plus:

- `symbols.json` -- detections with `id`, `class_name`, `bbox`, `confidence`, `method`
- `symbols_overlay.png` / `symbols_overlay_preview.png` -- bboxes color-coded by method (yellow-orange for template_match, blue for dinov2_nn)

Stage 10 (atom symbols -> typed signals) and Stage 11 (final merged JSON) are not yet built.